# Lecture 5.2: PostToolUse, Logging Every Tool Call

In this notebook you build one real hook. By the end, every tool call your agent makes is written to a log file that you can read afterwards, and you check that log against what really happened on disk.

**What you will do**

1. Create a small warehouse-notes project for the agent to work on.
2. Write an audit hook that records each tool call as one line of JSON.
3. Register the same hook on three events: `PreToolUse`, `PostToolUse` and `PostToolUseFailure`.
4. Read the log back, pair each request with how it finished using `tool_use_id`, and compare the log with the files on disk.
5. Use `additionalContext` in a second small hook to add information to a tool result that Claude sees.

**Cell numbers.** Every code cell has a number in the heading of the markdown cell above it, for example "Cell 8". The numbers match the recording guide, so you can follow along by number.

**A note on live output.** The agent chooses its own tool sequence, so the number and order of calls can differ from run to run. That is normal for a real agent. Focus on what each log entry means, not on matching counts exactly.

## Cell 1: Install the SDK

This cell installs `claude-agent-sdk`, the Python package that provides `query()`, `ClaudeAgentOptions` and `HookMatcher`. This is the only package the notebook needs.

| Part of the line | What it does |
|---|---|
| `!pip install` | Runs pip from a Colab cell. |
| `claude-agent-sdk==0.2.163` | Pins one exact version so every example behaves the way it does in the video. |
| `-q` | Keeps the install output quiet. |

If the package is already present in your current Colab session, pip checks the version and finishes quickly. If you want a newer version, the comment in the cell shows how, but newer versions may behave differently from the video.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.163 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.0 MB/s eta 0:00:00


## Cell 2: Set up your API key

The agent calls the Claude API, so it needs your Anthropic API key. In Colab, the safe way to provide it is Colab Secrets. The key stays in your account and never appears in the notebook.

**Add the secret (one time)**

1. Click the key icon in the left sidebar.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value. You create keys in the Claude Console at https://platform.claude.com/ under API Keys.
5. Switch **Notebook access** on.

The cell below reads the secret with `userdata.get` and writes it to the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

**Running locally instead of Colab?** Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before you start Python.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

Every agent in this notebook reads its model from one variable, `MODEL_NAME`. Changing this one line changes the model for the whole notebook.

We use Claude Haiku 4.5 here. It is fast and inexpensive, which suits a lecture where we run the agent several times. Model names change as new releases arrive, so the comment in the cell links to the models page where you can find the latest ones.

In [ ]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Cell 4: Import what we need

Three names cover this entire notebook.

| Name | Role |
|---|---|
| `query` | Starts the agent and returns an async stream of messages while the agent loop runs. |
| `ClaudeAgentOptions` | Holds the configuration: allowed tools, model, working directory and hooks. |
| `HookMatcher` | Attaches one or more hook callbacks to an event, optionally filtered by tool name. |

`HookMatcher` is the same class used to register hooks in the previous lecture.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher

## Cell 5: Create the sample project

The agent needs something to work on. This cell creates a small inbox with three warehouse notes under `/content/audit_demo/inbox`.

Sample files are created from code so the notebook does not depend on any upload. Notice that we create `note-a.txt`, `note-b.txt` and `note-c.txt` only. There will be no `note-z.txt`. That is deliberate, and you will see why when we ask the agent to read it.

In [ ]:
import os

base = "/content/audit_demo"
os.makedirs(f"{base}/inbox", exist_ok=True)

notes = {
    "note-a.txt": "Call the supplier about the delayed pallet.\n",
    "note-b.txt": "Renew the warehouse insurance before month end.\n",
    "note-c.txt": "Schedule the forklift inspection for Thursday.\n",
}
for name, text in notes.items():
    with open(f"{base}/inbox/{name}", "w") as f:
        f.write(text)

## Cell 6: Verify the files before any agent runs

The previous cell printed nothing, so we cannot yet tell what is on disk. This cell prints the folder tree and the contents of every note. It also checks `note-z.txt`.

Looking at the real files first gives us a ground truth to compare against later. The missing `note-z.txt` is a planned test: in a later cell the agent will try to read it, and we want to see how that failing call appears in the log.

In [ ]:
for root, dirs, files in os.walk(base):
    dirs.sort()
    level = root.replace(base, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for name in sorted(files):
        print(f"{indent}    {name}")

print()
for name in sorted(os.listdir(f"{base}/inbox")):
    with open(f"{base}/inbox/{name}") as f:
        print(f"{name}: {f.read().strip()}")

print()
print("note-z.txt exists:", os.path.exists(f"{base}/inbox/note-z.txt"))

audit_demo/
    inbox/
        note-a.txt
        note-b.txt
        note-c.txt

note-a.txt: Call the supplier about the delayed pallet.
note-b.txt: Renew the warehouse insurance before month end.
note-c.txt: Schedule the forklift inspection for Thursday.

note-z.txt exists: False


## Cell 7: Set up the log file

The audit log is a plain file at `/content/audit_demo/audit.jsonl`. The `.jsonl` extension signals JSON Lines: one JSON object per line. That format is easy to append to while the agent runs and easy to read back afterwards.

This cell also deletes any old log and any old digest. Without that reset, running the notebook twice would leave entries from the first run in the log, and a leftover digest could make the ground truth check pass for the wrong reason.

In [ ]:
import json
from datetime import datetime

LOG_PATH = "/content/audit_demo/audit.jsonl"
DIGEST_PATH = "/content/audit_demo/digest.txt"

for path in (LOG_PATH, DIGEST_PATH):
    if os.path.exists(path):
        os.remove(path)

## Cell 8: Write the audit hook (you type this one)

This is the first hook you write yourself. One function builds a record for each event and appends it to the log. It always returns `{}`, so it observes the agent without changing anything.

A hook callback receives three arguments: the input data, the `tool_use_id`, and a context object that is reserved for future use. Here is what each part of the record is for.

| Field | Why it is there |
|---|---|
| `time` | Lets you read the log in order. |
| `event` | Read from `hook_event_name`. We register this one function on three events, and this field is the only way to tell the entries apart. |
| `tool` and `input` | What the agent asked the tool to do. |
| `tool_use_id` | The same value `PreToolUse` saw for that call. It pairs "requested" with "finished". |
| `response` | Present only after a completed call. It has no fixed shape in the SDK types, so we cast it with `str()` and keep the first 150 characters. Large values such as file contents would otherwise flood the log. |
| `error` | Present on `PostToolUseFailure` entries. |

The write sits inside `try` and `except`. A logging hook should never be the reason an agent run breaks, so if the write fails we print a short message and carry on.

In [ ]:
async def audit_hook(input_data, tool_use_id, context):
    entry = {
        "time": datetime.now().isoformat(timespec="seconds"),
        "event": input_data["hook_event_name"],
        "tool": input_data.get("tool_name"),
        "tool_use_id": tool_use_id,
        "input": input_data.get("tool_input"),
    }
    if "tool_response" in input_data:
        entry["response"] = str(input_data["tool_response"])[:150]
    if "error" in input_data:
        entry["error"] = input_data["error"]
    try:
        with open(LOG_PATH, "a") as f:
            f.write(json.dumps(entry, default=str) + "\n")
    except Exception as e:
        print(f"[audit_hook] could not write log: {e}")
    return {}

## Cell 9: The task prompt

The prompt is long, so it is defined in its own cell. It states the goal and leaves the tool sequence to the agent. It asks for three things:

1. Read every note in the inbox using `Glob` and `Read`.
2. Create `digest.txt` with one line per note.
3. Try to read `note-z.txt`, the file we know does not exist, and say whether it exists.

The third request gives us a failing tool call to look for in the log.

In [ ]:
TASK_PROMPT = """Do these steps.
1. Use Glob to list the files in /content/audit_demo/inbox, then use Read to read every note you find.
2. Create /content/audit_demo/digest.txt with exactly one line per note, summarizing that note.
3. Also try to read /content/audit_demo/inbox/note-z.txt and tell me plainly whether it exists.
"""

## Cell 10: Register the hook on three events and run the agent (you type this one)

Now we hand the same `audit_hook` to the agent. The `hooks` option is a dictionary. Each key is an event name and each value is a list of `HookMatcher` objects.

| Piece | Meaning |
|---|---|
| `allowed_tools=["Read", "Write", "Glob"]` | Only the tools this task needs. |
| `model=MODEL_NAME` and `cwd="/content"` | The model variable from earlier and the working directory. |
| `HookMatcher(hooks=[audit_hook])` | No `matcher`, so the hook runs for every tool on that event. |
| Three keys, one function | `hook_event_name` inside each entry tells you which event produced it. |

`query()` returns an async stream of messages while the agent loop runs, so we use `async for` with top-level `await`. The final answer arrives on the message that has a `result` attribute.

`Write` is in `allowed_tools`. If `digest.txt` exists afterwards, that proves our hook never blocked the write.

In [ ]:
async for message in query(
    prompt=TASK_PROMPT,
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write", "Glob"],
        model=MODEL_NAME,
        cwd="/content",
        hooks={
            "PreToolUse": [HookMatcher(hooks=[audit_hook])],
            "PostToolUse": [HookMatcher(hooks=[audit_hook])],
            "PostToolUseFailure": [HookMatcher(hooks=[audit_hook])],
        },
    ),
):
    if hasattr(message, "result"):
        print(message.result)

All three steps are done.

1. **Inbox listing:** `/content/audit_demo/inbox` contains `note-a.txt`, `note-b.txt`, and `note-c.txt`. I read all three.

2. **Digest:** I created `/content/audit_demo/digest.txt` with one line per note:
   ```
   note-a: Call the supplier about the delayed pallet.
   note-b: Renew the warehouse insurance before month end.
   note-c: Schedule the forklift inspection for Thursday.
   ```

3. **note-z.txt:** It does not exist. The read failed with "File does not exist," and the inbox listing doesn't include it either.


## Cell 11: Read the log back

The log is only useful if you can read it. This cell loads every line of `audit.jsonl` into a list of dictionaries and prints one row per entry: the time, the event, the tool, and the last six characters of the `tool_use_id`.

Look at the event column. Your numbers and order may differ from someone else's run, because the agent decides which tools to call. Find the row for the `note-z.txt` read and notice which event it was logged under.

In [ ]:
def load_log():
    with open(LOG_PATH) as f:
        return [json.loads(line) for line in f]

entries = load_log()
for e in entries:
    print(f"{e['time']}  {e['event']:<19} {str(e['tool']):<6} id={str(e['tool_use_id'])[-6:]}")

2026-10-08T05:19:47  PreToolUse          Glob   id=M2PZ2Y
2026-10-08T05:19:47  PostToolUse         Glob   id=M2PZ2Y
2026-10-08T05:19:47  PreToolUse          Read   id=nBNrfN
2026-10-08T05:19:47  PostToolUse         Read   id=nBNrfN
2026-10-08T05:19:47  PreToolUse          Read   id=3XocwH
2026-10-08T05:19:47  PostToolUse         Read   id=3XocwH
2026-10-08T05:19:48  PreToolUse          Read   id=bUv4pe
2026-10-08T05:19:48  PostToolUse         Read   id=bUv4pe
2026-10-08T05:19:48  PreToolUse          Read   id=AEaNaN
2026-10-08T05:19:48  PostToolUseFailure  Read   id=AEaNaN
2026-10-08T05:19:50  PreToolUse          Write  id=mrbHHw
2026-10-08T05:19:50  PostToolUse         Write  id=mrbHHw


## Cell 12: Look at the real shape of tool_response

The SDK types give `tool_response` no fixed shape. For built-in tools it follows that tool's output schema. Rather than assume what it looks like, this cell prints the first logged response for each tool, plus any error text.

Read the output carefully. This is the evidence for what `Read`, `Glob` and `Write` actually hand to a post hook, truncated to 150 characters by our own hook.

In [ ]:
seen = set()
for e in entries:
    if "response" in e and e["tool"] not in seen:
        seen.add(e["tool"])
        print(e["tool"], "->", e["response"])
        print()

for e in entries:
    if "error" in e:
        print("FAILURE", e["tool"], "->", e["error"])

Glob -> {'filenames': ['audit_demo/inbox/note-c.txt', 'audit_demo/inbox/note-a.txt', 'audit_demo/inbox/note-b.txt'], 'durationMs': 18, 'numFiles': 3, 'truncat

Read -> {'type': 'text', 'file': {'filePath': '/content/audit_demo/inbox/note-a.txt', 'content': 'Call the supplier about the delayed pallet.\n', 'numLines': 

Write -> {'type': 'create', 'filePath': '/content/audit_demo/digest.txt', 'content': 'note-a: Call the supplier about the delayed pallet.\nnote-b: Renew the wa

FAILURE Read -> File does not exist. Note: your current working directory is /content.


## Cell 13: Pair each call with how it finished

`tool_use_id` is the same value for a call's `PreToolUse` entry and for the entry that records how it finished. This cell groups the logged events by that id and prints each call as a chain, for example `PreToolUse -> PostToolUse`.

Each line is one tool call. Compare the chain for the missing file with the chains for the successful calls.

In [ ]:
from collections import defaultdict

by_id = defaultdict(list)
for e in entries:
    by_id[e["tool_use_id"]].append(e["event"])
for call_id, events in by_id.items():
    print(str(call_id)[-6:], " -> ".join(events))

M2PZ2Y PreToolUse -> PostToolUse
nBNrfN PreToolUse -> PostToolUse
3XocwH PreToolUse -> PostToolUse
bUv4pe PreToolUse -> PostToolUse
AEaNaN PreToolUse -> PostToolUseFailure
mrbHHw PreToolUse -> PostToolUse


## Cell 14: Ground truth: check the log against the disk

A log records what the hook saw. The disk shows what really happened. This cell checks both: does `digest.txt` exist, and does the log contain a completed `Write` to that path?

We compare against the filesystem instead of parsing the message stream, because a file either exists or it does not. If the two ever disagree, trust the disk and investigate the hook.

In [ ]:
digest = "/content/audit_demo/digest.txt"
print("digest.txt exists:", os.path.exists(digest))
logged = any(
    e["event"] == "PostToolUse"
    and e["tool"] == "Write"
    and (e["input"] or {}).get("file_path") == digest
    for e in entries
)
print("Write to digest.txt logged as completed:", logged)

digest.txt exists: True
Write to digest.txt logged as completed: True


## Cell 15: Demo: adding context with additionalContext

So far our hook only observed. A post hook can also return `hookSpecificOutput` with `additionalContext`, which appends information to the tool result that Claude sees.

This second hook is registered on `PostToolUse` with `matcher="Read"`, so it runs only after `Read` calls. After each read it adds a style rule. Two details matter:

- `hookEventName` is copied from `input_data["hook_event_name"]`, so the output names the event it belongs to.
- The context is information for Claude, not a command. How the model uses it is up to the model.

The cell also deletes any old summary file so the check later is honest.

In [ ]:
SUMMARY_PATH = "/content/audit_demo/inbox-summary.txt"
if os.path.exists(SUMMARY_PATH):
    os.remove(SUMMARY_PATH)


async def style_hook(input_data, tool_use_id, context):
    return {
        "hookSpecificOutput": {
            "hookEventName": input_data["hook_event_name"],
            "additionalContext": "Style rule: begin every line of any summary you write with the word DRAFT:",
        }
    }

## Cell 16: Run the additionalContext demo

The task is small: read `note-a.txt` and write a one-line summary to `inbox-summary.txt`. The prompt never mentions the style rule. Only the hook does.

Only `Read` and `Write` are allowed, and the hook is attached to `PostToolUse` for `Read`.

In [ ]:
async for message in query(
    prompt="Read /content/audit_demo/inbox/note-a.txt and write a one-line summary of it to /content/audit_demo/inbox-summary.txt.",
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write"],
        model=MODEL_NAME,
        cwd="/content",
        hooks={
            "PostToolUse": [HookMatcher(matcher="Read", hooks=[style_hook])],
        },
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Done! I've read the note and created a summary. The file `/content/audit_demo/inbox/note-a.txt` contained "Call the supplier about the delayed pallet." I've written a one-line summary to `/content/audit_demo/inbox-summary.txt`:

**DRAFT: Note reminds to contact supplier regarding a delayed pallet delivery.**


## Cell 17: Ground truth for the demo

Check the file, not the chat. This cell confirms that the summary exists and prints whether its first line starts with `DRAFT:`.

Because the model decides how to use extra context, run the demo a few times if you like and watch for variation.

In [ ]:
if os.path.exists(SUMMARY_PATH):
    with open(SUMMARY_PATH) as f:
        first_line = f.read().strip().splitlines()[0]
    print("summary line:", first_line)
    print("starts with DRAFT:", first_line.startswith("DRAFT:"))
else:
    print("inbox-summary.txt does not exist")

summary line: DRAFT: Note reminds to contact supplier regarding a delayed pallet delivery.
starts with DRAFT: True


## Reference: returning async_ (not run)

A logging hook that only has a side effect can tell the agent not to wait for it. In Python you return `async_` because `async` is a reserved word. The SDK converts it to `async` when it sends the output to the CLI.

```python
async def fire_and_forget(input_data, tool_use_id, context):
    # start slow work in the background, then return at once
    return {"async_": True, "asyncTimeout": 30000}
```

| Field | Meaning |
|---|---|
| `async_` | `True` tells the agent to continue without waiting for the hook. |
| `asyncTimeout` | Timeout in milliseconds for the background work. |

**The trade-off:** the log may not be complete when the agent finishes. This notebook returns `{}` so the log is complete when the cell ends. An async output also cannot block, modify or add context, because the agent has already moved on.

**The rule to remember: log after the fact, prevent before the fact.** `PostToolUse` runs after the tool has acted and cannot undo it. To stop or change a call, use `PreToolUse`.

## Summary

You built a real hook.

1. `PostToolUse` receives `tool_name`, `tool_input`, `tool_response` and `tool_use_id`. `PostToolUseFailure` receives `error` instead of a response.
2. `tool_use_id` pairs each request with how it finished.
3. The audit hook writes one JSON object per line, truncates large values, catches its own exceptions and returns `{}`.
4. `additionalContext` adds information to a tool result that Claude sees.
5. The disk is the ground truth. Always check the log against it.

**Next lecture:** Lecture 5.3 covers changing what Claude sees from a tool, and stopping a run.